# DAY 3 v2 — Traditional ML Baseline: Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v9` (train=269,112 | val=3,926 | test=3,872)  
**Price unit:** `price = round(price_vnd / 1000)` — range 5–1000 (nghìn VND, same scale as English)  
**Primary metric:** MAE (k VND) — aligned with English pipeline  
**Training target:** raw `item.price` (5–1000), NO log transform — same as English day3  

## Nội dung notebook này (Section 0–4)

| Section | Nội dung |
|---|---|
| 0 | Setup & Load data |
| 1 | Statistical baselines (random, mean, median, category_mean) |
| 2 | Linear Regression baselines (simple features, BoW, TF-IDF char_wb) |
| 3 | Benchmark vectorizer: BoW vs char_wb vs Underthesea (50K subset + LGB default) |
| 4 | RandomForest + XGBoost + bảng so sánh tổng hợp |

## Section 0 — Setup & Load Data

In [1]:
import random
import sys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb

sys.path.insert(0, "..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Loaded {len(train):,} train | {len(val):,} val | {len(test):,} test")

Loaded 269,112 train | 3,926 val | 3,872 test


In [3]:
# EDA nhanh — hiểu phân phối giá trước khi model
train_prices = np.array([item.price for item in train])
print(f"Price range : {train_prices.min():.0f} – {train_prices.max():.0f} k VND")
print(f"Mean        : {train_prices.mean():.1f} k VND")
print(f"Median      : {np.median(train_prices):.1f} k VND")
print(f"Std         : {train_prices.std():.1f} k VND")
print()

print("Category distribution (train):")
for cat, cnt in Counter(item.category for item in train).most_common():
    print(f"  {cat:35s}: {cnt:,} ({cnt/len(train)*100:.1f}%)")
print()

summary_lens = [len(item.summary) for item in train]
print(f"Summary length: mean={np.mean(summary_lens):.0f} | min={min(summary_lens)} | max={max(summary_lens)}")
print()

print("Sample item:")
item0 = train[0]
print(f"  title   : {item0.title}")
print(f"  category: {item0.category}")
print(f"  brand   : {item0.brand}")
print(f"  price   : {item0.price} (k VND)")
print(f"  summary :\n{item0.summary}")

Price range : 5 – 1000 k VND
Mean        : 330.5 k VND
Median      : 215.0 k VND
Std         : 268.8 k VND

Category distribution (train):
  Thời Trang                         : 84,383 (31.4%)
  Nhà Cửa - Đời Sống                 : 54,409 (20.2%)
  Điện Tử - Công Nghệ                : 49,099 (18.2%)
  Làm Đẹp - Sức Khỏe                 : 36,903 (13.7%)
  Mẹ và Bé                           : 17,244 (6.4%)
  Bách Hóa                           : 16,898 (6.3%)
  Điện Lạnh và Gia Dụng              : 6,151 (2.3%)
  Ô Tô - Xe Máy                      : 4,025 (1.5%)

Summary length: mean=323 | min=176 | max=3184

Sample item:
  title   : Ốp lưng dành cho Xiaomi Redmi Note 11 4G mẫu Quỳnh Ế
  category: Điện Tử - Công Nghệ
  brand   : OLSG
  price   : 55 (k VND)
  summary :
Tiêu đề: Ốp lưng silicone dẻo + nhựa cứng cho Xiaomi Redmi Note 11
Danh mục: Phụ kiện điện thoại
Thương hiệu: OLSG
Mô tả: Ốp lưng dành cho Xiaomi Redmi Note 11 4G với viền ốp dẻo và nhựa cứng, bảo vệ toàn diện máy khỏi trầy x

In [4]:
# Dict chứa kết quả tất cả models — dùng để so sánh cuối Section 4
results = {}

## Section 1 — Statistical Baselines

Không dùng NLP — chỉ là heuristic đơn giản.  
Đây là "đường sàn" mà mọi ML model đều phải vượt qua.

Evaluate trên `test` set, 200 samples — giống English day3.

In [5]:
def random_pricer(item):
    return random.randrange(5, 1001)

random.seed(42)
results["1a. Random"] = evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

180 20 4 404 160 430 51 194 709 39 428 518 391 511 230 420 307 112 75 123 206 707 299 456 68 280 12 623 145 604 392 355 162 275 532 200 693 726 883 53 535 56 269 299 263 491 114 60 899 637 286 256 11 585 29 153 225 153 298 125 6 171 639 390 354 93 690 124 495 495 70 735 459 188 722 768 175 397 148 425 169 639 432 452 747 107 63 781 134 793 441 195 136 421 556 670 3 629 305 228 9 512 50 424 401 482 418 40 438 435 163 340 248 85 283 418 14 568 848 545 268 433 0 610 239 597 502 700 792 69 416 11 330 273 33 140 247 278 608 700 205 845 490 272 203 621 217 59 755 876 15 421 748 423 60 542 628 20 385 206 489 534 208 691 473 345 76 240 5 464 82 147 349 132 106 243 384 592 522 199 626 268 465 85 192 305 336 334 247 27 549 32 561 771 123 444 654 698 584 841 


In [6]:
training_average = sum(item.price for item in train) / len(train)
print(f"Training average price: {training_average:.1f} k VND")

def constant_pricer(item):
    return training_average

results["1b. Constant (Mean)"] = evaluate(constant_pricer, test)

Training average price: 330.5 k VND


  0%|          | 0/200 [00:00<?, ?it/s]

149 231 304 30 204 355 46 11 280 182 61 85 197 278 6 141 200 181 220 107 104 620 107 165 230 31 134 215 485 211 159 251 262 141 254 241 190 161 564 505 35 106 120 192 240 450 280 165 244 181 267 35 242 649 255 111 317 126 5 180 495 110 217 245 130 105 71 186 250 255 95 211 141 120 141 211 220 131 19 29 85 360 80 360 281 78 181 231 221 231 219 131 177 560 231 141 45 470 251 190 120 151 102 31 233 108 80 292 140 110 13 119 669 160 125 270 49 519 226 215 111 188 101 234 246 59 41 231 524 160 100 282 169 275 52 191 11 169 2 60 50 273 80 275 95 275 31 139 175 265 318 277 569 369 243 469 191 269 159 262 50 261 95 97 201 262 31 139 189 660 98 85 99 145 450 30 215 275 360 287 255 161 180 175 123 240 51 488 31 95 120 209 5 131 191 73 241 490 211 260 


In [7]:
training_median = float(np.median(train_prices))
print(f"Training median price: {training_median:.1f} k VND")

def median_pricer(item):
    return training_median

results["1c. Median"] = evaluate(median_pricer, test)

Training median price: 215.0 k VND


  0%|          | 0/200 [00:00<?, ?it/s]

264 116 189 145 89 470 69 126 165 67 54 30 312 163 109 26 85 66 105 8 219 735 8 50 115 84 19 100 600 96 44 136 147 26 139 126 75 46 679 620 80 9 235 77 125 565 165 50 129 66 152 150 127 764 140 4 432 11 110 65 610 5 102 130 15 10 44 301 365 140 20 96 26 235 26 96 335 16 134 86 30 475 35 475 166 193 66 116 106 116 334 16 62 675 116 26 160 585 136 75 5 36 13 84 348 7 35 177 25 5 128 4 784 45 240 155 164 634 111 100 226 73 14 119 361 56 74 116 639 45 215 167 284 160 167 76 104 284 117 55 165 158 35 160 210 160 84 254 60 150 433 162 684 484 128 584 76 384 44 147 65 146 20 18 86 147 84 254 74 775 213 30 214 30 565 145 330 160 475 402 140 46 65 60 8 125 64 603 84 20 235 324 110 16 76 42 126 605 96 145 


In [8]:
cat_price_sums = defaultdict(float)
cat_counts_map = defaultdict(int)
for item in train:
    cat_price_sums[item.category] += item.price
    cat_counts_map[item.category] += 1
cat_means = {cat: cat_price_sums[cat] / cat_counts_map[cat] for cat in cat_price_sums}

print("Category mean prices (k VND):")
for cat, mean_p in sorted(cat_means.items(), key=lambda x: x[1]):
    print(f"  {cat:35s}: {mean_p:.1f}")

def category_mean_pricer(item):
    return cat_means.get(item.category, training_average)

results["1d. Category Mean"] = evaluate(category_mean_pricer, test)

Category mean prices (k VND):
  Bách Hóa                           : 210.6
  Thời Trang                         : 291.6
  Làm Đẹp - Sức Khỏe                 : 304.5
  Mẹ và Bé                           : 309.5
  Ô Tô - Xe Máy                      : 339.2
  Nhà Cửa - Đời Sống                 : 365.8
  Điện Tử - Công Nghệ                : 401.9
  Điện Lạnh và Gia Dụng              : 519.5


  0%|          | 0/200 [00:00<?, ?it/s]

113 303 278 42 276 319 118 61 352 63 133 47 161 314 32 22 272 143 292 69 142 548 69 145 192 7 108 177 505 247 168 225 236 103 234 313 152 233 492 315 71 86 48 228 202 476 161 127 218 217 247 37 278 669 264 73 281 88 33 216 521 90 98 207 11 85 33 114 288 327 69 185 103 239 103 283 258 203 57 9 157 398 89 380 243 98 143 193 257 193 257 93 249 598 193 121 71 398 213 262 156 131 174 7 161 70 116 254 212 72 23 191 479 134 53 151 87 447 200 287 39 197 63 196 174 131 3 267 488 134 126 244 189 347 16 165 47 195 40 34 76 309 152 347 86 237 11 177 149 227 282 239 533 297 124 433 263 295 121 298 86 297 24 169 175 298 103 103 261 588 62 157 119 107 378 56 253 347 386 215 229 197 252 211 195 249 13 452 67 167 140 137 33 203 153 145 277 418 173 222 


## Section 2 — Linear Regression Baselines

Train trực tiếp trên raw price (5–1000) — giống English day3.  
Không dùng log transform vì price range đã align với English.

In [5]:
# Prepare targets và documents
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
print(f"Train prices: {len(prices):,} items, range {prices.min():.0f}–{prices.max():.0f}")

Train prices: 269,112 items, range 5–1000


In [10]:
# 2a. LR với features đơn giản: text_length, title_length
# English dùng weight + text_length; ta không có weight nên dùng text_length + title_length

def get_features(item):
    return {
        "text_length": len(item.summary),
        "title_length": len(item.title),
    }

def list_to_df(items):
    feats = [get_features(item) for item in items]
    df = pd.DataFrame(feats)
    df["price"] = [item.price for item in items]
    return df

train_df = list_to_df(train)
test_df = list_to_df(test)

feat_cols = ["text_length", "title_length"]

np.random.seed(42)
lr_simple = LinearRegression()
lr_simple.fit(train_df[feat_cols], train_df["price"])

for feat, coef in zip(feat_cols, lr_simple.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  Intercept: {lr_simple.intercept_:.4f}")

mse = mean_squared_error(test_df["price"], lr_simple.predict(test_df[feat_cols]))
r2 = r2_score(test_df["price"], lr_simple.predict(test_df[feat_cols]))
print(f"  MSE: {mse:,.0f}  R²: {r2:.4f}")

def linear_regression_pricer(item):
    feats = pd.DataFrame([get_features(item)])
    return max(5, lr_simple.predict(feats)[0])

results["2a. LR Simple Features"] = evaluate(linear_regression_pricer, test)

  text_length: 0.5313
  title_length: 0.4540
  Intercept: 125.8050
  MSE: 52,893  R²: -0.0020


  0%|          | 0/200 [00:00<?, ?it/s]

132 206 255 27 176 322 3 18 258 216 69 48 230 240 17 131 238 157 194 83 149 637 59 156 194 58 179 161 485 170 133 269 294 115 231 206 158 214 530 515 41 87 4 154 218 426 237 153 227 149 234 39 216 654 251 118 370 120 53 213 494 57 151 239 160 74 69 210 279 272 84 197 112 141 105 235 252 132 6 20 41 405 44 351 241 77 137 220 179 229 214 109 88 613 168 132 79 481 225 146 78 173 135 29 231 74 82 298 138 78 42 104 699 116 134 225 67 538 216 216 124 173 100 211 257 62 10 198 563 144 91 247 209 264 82 194 11 155 6 61 69 230 97 268 37 275 50 168 178 251 363 239 587 384 230 524 174 274 162 253 46 239 86 90 190 277 20 156 270 641 101 65 115 128 456 45 255 254 332 285 217 153 211 183 107 193 21 494 9 97 114 224 24 116 143 49 253 522 228 220 


In [11]:
# 2b. LR + CountVectorizer (Bag-of-Words, 2000 features) — giống English day3

np.random.seed(42)
vec_bow = CountVectorizer(max_features=2000)
X_bow = vec_bow.fit_transform(documents)
print(f"BoW vocabulary size: {len(vec_bow.get_feature_names_out())}")
print(f"Sample words: {vec_bow.get_feature_names_out()[1000:1010]}")

lr_bow = LinearRegression()
lr_bow.fit(X_bow, prices)

def natural_language_linear_regression_pricer(item):
    x = vec_bow.transform([item.summary])
    return max(5, lr_bow.predict(x)[0])

results["2b. LR + BoW"] = evaluate(natural_language_linear_regression_pricer, test)

BoW vocabulary size: 2000
Sample words: ['model' 'mr' 'msi' 'mua' 'muối' 'muỗi' 'my' 'm²' 'mà' 'mài']


  0%|          | 0/200 [00:00<?, ?it/s]

7 105 201 161 1 309 97 84 377 64 19 33 82 173 131 73 121 343 46 2 92 280 160 34 75 123 95 85 563 141 161 269 180 117 44 193 112 52 479 470 202 13 65 103 283 353 10 15 42 30 191 156 220 421 219 3 27 80 2 147 263 57 66 109 15 22 29 37 128 14 37 114 96 119 76 21 45 114 91 42 50 389 125 198 160 93 26 110 269 94 72 74 170 574 140 94 91 368 212 210 118 74 55 227 121 45 118 33 7 22 177 72 371 211 18 62 55 457 45 2 114 73 40 82 76 145 9 46 459 49 5 93 165 51 90 130 160 5 32 64 65 332 59 5 90 157 19 134 134 181 156 263 166 457 52 248 196 12 34 51 52 296 125 229 134 87 3 125 91 497 93 172 12 55 3 98 161 39 166 108 367 98 141 145 29 379 147 445 274 91 284 59 4 38 86 20 118 278 49 5 


In [6]:
from sklearn.linear_model import Ridge

vec_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_tfidf = vec_tfidf.fit_transform(documents)
print(f"TF-IDF char_wb features: {X_tfidf.shape}")

lr_tfidf = Ridge(alpha=1.0, solver="sag", max_iter=300)
lr_tfidf.fit(X_tfidf, prices)
print("Ridge fit done.")

def tfidf_lr_pricer(item):
    x = vec_tfidf.transform([item.summary])
    return max(5, lr_tfidf.predict(x)[0])

results["2c. Ridge + TF-IDF char_wb"] = evaluate(tfidf_lr_pricer, test)

TF-IDF char_wb features: (269112, 100000)
Ridge fit done.


  0%|          | 0/200 [00:00<?, ?it/s]

5 35 129 199 53 113 160 52 281 33 89 11 150 204 105 17 36 366 31 39 126 26 82 11 33 103 157 10 664 131 92 14 99 32 28 128 53 162 527 461 180 88 90 133 306 414 45 26 7 50 55 99 149 342 2 15 97 30 62 253 90 22 108 96 17 59 23 10 30 10 56 13 87 195 43 155 131 66 23 88 161 495 219 123 155 36 2 86 101 94 35 95 107 308 51 47 8 328 168 114 195 145 166 275 171 94 130 17 8 48 162 87 292 294 151 35 60 203 22 116 202 67 41 68 52 15 36 236 211 29 66 107 136 44 44 135 204 123 30 25 74 246 66 50 85 168 62 25 114 171 164 223 197 73 89 199 92 10 80 103 97 112 171 48 54 223 124 191 58 308 30 117 93 45 37 6 119 17 117 63 134 6 4 180 11 43 100 376 331 74 170 73 28 38 80 14 84 124 114 27 


## Section 3 — Benchmark Vectorizer

So sánh 3 vectorizer với LGB default trên **50K subset** để chọn vectorizer tốt nhất cho Section 4.

| # | Vectorizer | Ghi chú |
|---|---|---|
| A | `CountVectorizer` — Bag-of-Words | Baseline nhanh nhất |
| B | `TfidfVectorizer(char_wb, 2-4gram)` | **Khuyến nghị chính** |
| C | `Underthesea + TfidfVectorizer(word)` | Chậm (~20-30 phút cho 50K) |

In [6]:
import psutil
mem = psutil.virtual_memory()
print(f"Total RAM : {mem.total/1e9:.1f} GB")
print(f"Dang dung : {mem.used/1e9:.1f} GB")
print(f"Con lai   : {mem.available/1e9:.1f} GB")

Total RAM : 30.0 GB
Dang dung : 9.4 GB
Con lai   : 20.6 GB


In [7]:
# Cài underthesea nếu chưa có
import subprocess
subprocess.run(["uv", "add", "underthesea"], check=True)

CompletedProcess(args=['uv', 'add', 'underthesea'], returncode=0)

In [11]:
# 50K subset cho benchmark
BENCH_SIZE = 50_000
train_sub = train[:BENCH_SIZE]
docs_sub = [item.summary for item in train_sub]
prices_sub = np.array([float(item.price) for item in train_sub])
print(f"Benchmark subset: {BENCH_SIZE:,} items, price range {prices_sub.min():.0f}–{prices_sub.max():.0f}")

Benchmark subset: 50,000 items, price range 5–1000


In [13]:
# 3A. BoW + LGB default
print("[3A] BoW + LGB...")
vec_bow_b = CountVectorizer(max_features=2000)
X_bow_b = vec_bow_b.fit_transform(docs_sub).astype(np.float32)

lgb_bow = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_bow.fit(X_bow_b, prices_sub)

def bow_lgb_bench(item):
    x = vec_bow_b.transform([item.summary]).astype(np.float32)
    return max(5, lgb_bow.predict(x)[0])

print("Evaluating [3A] BoW + LGB (200 test samples):")
results["3A. Bench BoW + LGB"] = evaluate(bow_lgb_bench, test)

[3A] BoW + LGB...
Evaluating [3A] BoW + LGB (200 test samples):


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

116 3 154 18 4 179 94 77 309 47 42 18 169 236 3 1 77 203 20 74 184 233 105 3 233 62 159 25 595 63 68 122 118 155 60 124 15 45 551 409 126 14 162 57 248 288 10 47 119 4 239 13 201 399 129 34 74 14 31 125 249 62 46 121 80 12 145 56 63 21 49 147 65 64 86 42 79 20 25 91 62 359 38 369 87 222 93 65 182 13 64 87 140 561 179 50 120 303 125 101 175 23 96 151 88 92 12 33 12 14 103 41 451 317 37 47 19 465 74 83 81 89 4 103 89 40 7 73 277 57 149 47 172 34 58 219 176 67 54 55 5 201 118 18 11 181 206 10 38 152 258 219 57 366 163 152 128 94 8 83 28 227 102 290 154 114 7 84 42 359 98 53 77 120 52 102 123 76 247 95 344 14 83 105 24 208 58 573 352 24 253 53 26 27 57 85 203 260 8 48 


In [14]:
# 3B. char_wb TF-IDF + LGB default
print("[3B] char_wb TF-IDF + LGB...")
vec_cw_b = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 4),
    max_features=100_000, sublinear_tf=True,
)
X_cw_b = vec_cw_b.fit_transform(docs_sub)
print(f"  Feature matrix: {X_cw_b.shape}")

lgb_cw = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_cw.fit(X_cw_b, prices_sub)

def charwb_lgb_bench(item):
    x = vec_cw_b.transform([item.summary])
    return max(5, lgb_cw.predict(x)[0])

print("Evaluating [3B] char_wb + LGB (200 test samples):")
results["3B. Bench char_wb + LGB"] = evaluate(charwb_lgb_bench, test)

[3B] char_wb TF-IDF + LGB...
  Feature matrix: (50000, 100000)
Evaluating [3B] char_wb + LGB (200 test samples):


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

2 73 137 184 3 84 181 45 205 60 14 26 244 146 54 102 125 191 23 16 198 142 26 25 163 86 99 50 609 106 114 63 87 42 40 5 57 57 528 453 148 4 138 8 201 284 58 4 28 100 109 20 163 290 91 101 90 9 131 109 238 28 2 65 84 43 15 75 149 44 51 39 16 36 16 173 207 28 2 13 38 430 120 356 109 148 60 30 92 17 40 84 109 394 19 51 13 296 64 113 281 22 89 139 151 84 51 34 10 57 218 61 414 228 116 65 172 330 58 63 20 33 70 11 61 20 31 144 202 18 31 63 170 7 21 316 149 149 118 51 85 205 82 6 116 98 48 45 97 46 328 116 177 65 192 283 159 82 11 6 36 293 150 115 70 100 20 41 94 450 16 68 51 125 11 103 109 53 205 136 216 17 169 111 88 183 63 514 347 70 240 4 51 86 27 20 36 251 57 62 


In [17]:
# 3B. char_wb TF-IDF + LGB default
print("[3B] char_wb TF-IDF + LGB...")
vec_cw_b = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 4),
    max_features=100_000, sublinear_tf=True,
)
X_cw_b = vec_cw_b.fit_transform(docs_sub)
print(f"  Feature matrix: {X_cw_b.shape}")

lgb_cw = lgb.LGBMRegressor(
    n_estimators=1000, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_cw.fit(X_cw_b, prices_sub)

def charwb_lgb_bench(item):
    x = vec_cw_b.transform([item.summary])
    return max(5, lgb_cw.predict(x)[0])

print("Evaluating [3B] char_wb + LGB (200 test samples):")
results["3B. Bench char_wb + LGB"] = evaluate(charwb_lgb_bench, test)

[3B] char_wb TF-IDF + LGB...
  Feature matrix: (50000, 100000)
Evaluating [3B] char_wb + LGB (200 test samples):


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

2 128 116 179 4 107 192 55 179 52 19 9 200 200 55 104 153 161 76 11 187 98 22 21 137 116 116 22 589 112 93 87 47 52 63 15 81 57 500 393 137 27 170 47 204 308 35 1 30 117 78 4 133 297 68 95 76 37 155 113 203 32 16 97 78 1 5 89 129 35 67 7 4 9 26 221 212 44 4 2 49 456 137 326 103 133 57 25 67 11 40 41 119 371 9 47 0 299 55 68 309 63 103 173 107 77 68 40 39 56 215 37 407 254 78 57 140 351 56 59 4 44 81 45 54 18 66 170 201 33 12 35 186 12 12 306 156 159 133 44 133 185 58 10 86 83 31 43 99 41 310 119 190 85 190 237 164 49 18 7 34 296 141 79 58 100 27 34 82 445 28 73 28 108 26 103 136 34 182 127 220 13 177 106 88 182 59 484 329 64 205 10 16 74 27 16 25 209 48 33 


In [16]:
# 3C. Underthesea + TF-IDF word + LGB default
# LƯU Ý: tokenize 50K docs mất khoảng 20-30 phút
from underthesea import word_tokenize

def vi_tokenize(text):
    return word_tokenize(text, format="text")

print("[3C] Underthesea tokenizing 50K docs (~20-30 min)...")
docs_sub_vi = [vi_tokenize(doc) for doc in docs_sub]
print("Tokenization done.")

vec_vi_b = TfidfVectorizer(max_features=100_000, sublinear_tf=True)
X_vi_b = vec_vi_b.fit_transform(docs_sub_vi).astype(np.float32)
print(f"  Feature matrix: {X_vi_b.shape}")

lgb_vi = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_vi.fit(X_vi_b, prices_sub)

def under_lgb_bench(item):
    tok = vi_tokenize(item.summary)
    x = vec_vi_b.transform([tok]).astype(np.float32)
    return max(5, lgb_vi.predict(x)[0])

print("Evaluating [3C] Underthesea + LGB (200 test samples):")
results["3C. Bench Underthesea + LGB"] = evaluate(under_lgb_bench, test)

[3C] Underthesea tokenizing 50K docs (~20-30 min)...
Tokenization done.
  Feature matrix: (50000, 94512)
Evaluating [3C] Underthesea + LGB (200 test samples):


c:\hieu\DACN3\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



  0%|          | 0/200 [00:00<?, ?it/s]

178 36 139 82 103 261 14 74 210 124 4 53 197 104 113 72 155 126 23 8 162 266 63 67 162 7 122 7 594 49 144 177 41 32 30 59 58 26 619 444 79 13 179 35 28 263 4 40 54 46 50 21 165 396 123 59 77 49 61 111 330 58 29 134 126 17 39 50 120 43 39 222 8 132 79 26 44 13 12 171 40 428 37 221 139 177 22 31 129 36 42 22 65 363 31 28 23 187 136 335 134 17 63 180 190 67 1 34 24 5 60 85 373 243 37 42 157 454 4 54 40 42 106 29 15 6 13 88 385 4 135 41 255 27 85 91 177 4 87 180 67 214 19 36 47 138 122 181 80 106 194 117 207 472 152 254 5 192 10 20 35 285 123 204 155 95 168 146 29 435 13 52 53 95 95 232 276 22 257 51 279 160 27 44 42 239 44 606 363 22 170 49 32 55 19 66 158 331 55 95 


In [18]:
# Bảng so sánh 3 vectorizer
print("=" * 65)
print("VECTORIZER BENCHMARK (50K train subset, LGB default, 200 test)")
print("=" * 65)
bench_keys = ["3A. Bench BoW + LGB", "3B. Bench char_wb + LGB", "3C. Bench Underthesea + LGB"]
for name in bench_keys:
    m = results[name]
    print(f"{name:35s}  MAE={m['mae']:.1f}k  MSE={m['mse']:,.0f}  R²={m['r2']:.1f}%")
print("=" * 65)
best_vec = min(bench_keys, key=lambda k: results[k]["mae"])
print(f"Best vectorizer (MAE): {best_vec}")

VECTORIZER BENCHMARK (50K train subset, LGB default, 200 test)
3A. Bench BoW + LGB                  MAE=120.8k  MSE=28,226  R²=50.8%
3B. Bench char_wb + LGB              MAE=108.8k  MSE=23,173  R²=59.6%
3C. Bench Underthesea + LGB          MAE=118.0k  MSE=28,239  R²=50.8%
Best vectorizer (MAE): 3B. Bench char_wb + LGB


In [19]:
import json
with open("day3_v2_results_partial.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Saved:", list(results.keys()))

Saved: ['2c. Ridge + TF-IDF char_wb', '3A. Bench BoW + LGB', '3B. Bench char_wb + LGB', '3C. Bench Underthesea + LGB']


## Section 4 — RandomForest & XGBoost

Dùng vectorizer tốt nhất từ Section 3 (kỳ vọng: `char_wb`), refit trên **toàn bộ train** (269K).

- **RandomForest**: 15K subset (slow với 269K) — giống English day3  
- **XGBoost**: full 269K — giống English day3 (có thể mất 30-60 phút)

In [7]:
# Refit char_wb vectorizer trên full 269K train
# Đổi sang BoW nếu Section 3 benchmark cho thấy BoW tốt hơn
print("Fitting char_wb TF-IDF on all 269K training docs...")
vec_main = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_main = vec_main.fit_transform(documents)
print(f"Feature matrix: {X_main.shape}")

Fitting char_wb TF-IDF on all 269K training docs...
Feature matrix: (269112, 100000)


In [8]:
# 4a. Random Forest — 15K subset
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_main[:subset], prices[:subset])

def random_forest(item):
    x = vec_main.transform([item.summary])
    return max(5, rf_model.predict(x)[0])

results["4a. RandomForest (15K)"] = evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

202 70 217 39 97 171 8 26 227 161 46 11 339 156 67 33 38 98 130 19 204 145 57 99 246 195 52 12 566 196 79 175 144 73 50 110 60 104 673 550 109 38 38 83 248 319 166 8 77 65 212 143 165 496 155 19 249 7 76 129 442 25 133 91 39 57 64 57 224 60 32 179 1 8 52 154 54 10 20 31 5 356 46 371 82 122 74 12 112 90 141 96 108 604 70 165 133 231 92 237 56 81 6 32 209 4 45 122 48 38 72 43 627 147 21 148 97 293 52 66 27 59 1 10 68 87 18 132 266 68 109 50 225 1 63 111 56 296 32 36 54 253 14 3 91 136 72 118 144 52 236 151 251 163 95 337 89 290 37 162 120 137 53 192 109 149 0 85 136 628 53 98 87 156 48 171 13 27 324 32 344 150 155 98 34 372 42 475 231 120 175 30 25 87 6 57 139 299 97 95 


In [9]:
# 4b. XGBoost — full 269K
# colsample_bytree=0.3: sample 30K/100K features moi tree → nhanh hon 2-3x
np.random.seed(42)
xgb_model = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1,
    learning_rate=0.1,
    tree_method="hist",
    colsample_bytree=0.3,
    subsample=0.8,
    max_depth=6,
)   
xgb_model.fit(X_main, prices)

def xg_boost(item):
    x = vec_main.transform([item.summary])
    return max(5, xgb_model.predict(x)[0])
    
results["4b. XGBoost"] = evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

142 52 135 67 46 248 86 27 236 178 24 46 254 28 107 19 32 62 76 3 198 297 50 43 153 13 5 2 584 147 100 235 141 36 62 127 8 83 630 516 98 27 69 69 172 366 125 17 80 13 152 69 229 536 142 3 55 23 2 134 333 33 42 72 89 20 3 79 209 112 64 230 74 45 55 101 16 37 14 71 23 420 84 397 116 116 22 67 127 70 161 75 130 490 65 23 27 327 78 216 118 51 22 3 224 17 46 42 17 19 20 2 545 173 101 171 102 294 28 30 24 69 29 56 57 72 39 152 394 32 159 105 231 16 43 157 8 84 16 29 2 169 4 9 142 185 39 199 63 50 387 201 229 178 159 420 75 237 44 129 39 218 45 264 128 201 3 32 209 654 14 108 67 94 118 190 165 98 251 39 276 24 245 68 24 323 92 460 288 15 215 65 6 34 12 91 166 353 99 110 


In [ ]:
# === BẢNG SO SÁNH TỔNG HỢP — Section 0–4 ===
print("=" * 70)
print("LEADERBOARD — Day 3 v2 (Section 0-4, 200 test samples)")
print("=" * 70)
print(f"{'Model':35s}  {'MAE (k VND)':>12}  {'MSE':>10}  {'R²':>7}")
print("-" * 70)
for name, m in results.items():
    print(f"{name:35s}  {m['mae']:>9.1f}k     {m['mse']:>10,.0f}  {m['r2']:>6.1f}%")
print("=" * 70)
best = min(results, key=lambda k: results[k]["mae"])
print(f"Best so far: {best}  MAE={results[best]['mae']:.1f}k VND")
print()
print("Day 3 v1 tham chieu: MAE ~110k VND")